# Credit Card Fraud Detection - XGBoost

This notebook trains a supervised XGBoost classifier as a stronger benchmark against Logistic Regression and the unsupervised Autoencoder. XGBoost is not an unsupervised method; it uses the `Class` label during training.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "creditcard_2023.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "creditcard_2023.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(RESULTS_DIR / ".matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from xgboost import XGBClassifier

RANDOM_STATE = 42

## Load and prepare data

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.dropna()

X = df.drop(columns=["id", "Class"])
y = df["Class"]

print(df.shape)
print(y.value_counts())

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print("Train:", X_train.shape, y_train.value_counts().to_dict())
print("Validation:", X_val.shape, y_val.value_counts().to_dict())
print("Test:", X_test.shape, y_test.value_counts().to_dict())

## Train XGBoost classifier

In [ ]:
neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_pos_weight = neg_count / pos_count

xgb_model = XGBClassifier(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.90,
    colsample_bytree=0.90,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    scale_pos_weight=scale_pos_weight,
)

xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_model

## Tune decision threshold on validation set

In [ ]:
y_val_proba = xgb_model.predict_proba(X_val)[:, 1]
y_test_proba = xgb_model.predict_proba(X_test)[:, 1]

thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
threshold_rows = []

for threshold in thresholds:
    y_val_pred = (y_val_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_val_pred).ravel()

    threshold_rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_val, y_val_pred),
        "precision_fraud": precision_score(y_val, y_val_pred, zero_division=0),
        "recall_fraud": recall_score(y_val, y_val_pred, zero_division=0),
        "f1_fraud": f1_score(y_val, y_val_pred, zero_division=0),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
    })

xgboost_threshold_results = pd.DataFrame(threshold_rows)
xgboost_threshold_results.to_csv(RESULTS_DIR / "xgboost_threshold_results.csv", index=False)

best_threshold_row = xgboost_threshold_results.sort_values(
    ["f1_fraud", "recall_fraud"],
    ascending=False,
).iloc[0]
best_threshold = float(best_threshold_row["threshold"])

print("Best validation threshold:", best_threshold)
xgboost_threshold_results

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(xgboost_threshold_results["threshold"], xgboost_threshold_results["precision_fraud"], marker="o", label="Precision")
plt.plot(xgboost_threshold_results["threshold"], xgboost_threshold_results["recall_fraud"], marker="o", label="Recall")
plt.plot(xgboost_threshold_results["threshold"], xgboost_threshold_results["f1_fraud"], marker="o", label="F1-score")
plt.axvline(best_threshold, linestyle="--", color="black", label=f"Best threshold = {best_threshold:.2f}")
plt.xlabel("Decision threshold")
plt.ylabel("Validation score")
plt.title("XGBoost Threshold Tuning")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "xgboost_threshold_tuning.png", dpi=300, bbox_inches="tight")
plt.show()

## Evaluate on test set

In [ ]:
y_test_pred = (y_test_proba >= best_threshold).astype(int)
cm = confusion_matrix(y_test, y_test_pred)
report = classification_report(y_test, y_test_pred)

print(cm)
print(report)

with open(RESULTS_DIR / "xgboost_classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Fraud", "Fraud"],
)
disp.plot()
plt.title("Confusion Matrix - XGBoost")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix_xgboost.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
xgboost_metrics = pd.DataFrame({
    "Model": ["XGBoost (best F1 threshold)"],
    "Accuracy": [accuracy_score(y_test, y_test_pred)],
    "Precision": [precision_score(y_test, y_test_pred, zero_division=0)],
    "Recall": [recall_score(y_test, y_test_pred, zero_division=0)],
    "F1-score": [f1_score(y_test, y_test_pred, zero_division=0)],
    "ROC-AUC": [roc_auc_score(y_test, y_test_proba)],
    "PR-AUC": [average_precision_score(y_test, y_test_proba)],
    "Best Threshold Percentile": [np.nan],
    "Threshold": [best_threshold],
})

xgboost_metrics.to_csv(RESULTS_DIR / "xgboost_metrics.csv", index=False)
xgboost_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y_test, y_test_proba, ax=axes[0], name="XGBoost")
PrecisionRecallDisplay.from_predictions(y_test, y_test_proba, ax=axes[1], name="XGBoost")
axes[0].set_title("ROC Curve - XGBoost")
axes[1].set_title("Precision-Recall Curve - XGBoost")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "xgboost_roc_pr_curves.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)

feature_importance.to_csv(RESULTS_DIR / "xgboost_feature_importance.csv", index=False)

top_features = feature_importance.head(15).sort_values("importance")
plt.figure(figsize=(8, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Importance")
plt.title("Top 15 XGBoost Feature Importances")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "xgboost_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

feature_importance.head(15)

## Update final model comparison

In [ ]:
comparison_path = RESULTS_DIR / "model_comparison.csv"
if comparison_path.exists():
    model_comparison = pd.read_csv(comparison_path)
else:
    model_comparison = pd.DataFrame()

model_comparison = model_comparison[
    ~model_comparison.get("Model", pd.Series(dtype=str)).astype(str).str.contains("XGBoost", na=False)
]

model_comparison = pd.concat(
    [model_comparison, xgboost_metrics],
    ignore_index=True,
    sort=False,
)
model_comparison.to_csv(comparison_path, index=False)
model_comparison

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"]
plot_df = model_comparison.set_index("Model")[metrics_to_plot]

ax = plot_df.T.plot(kind="bar", figsize=(11, 6))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Fraud Detection Model Comparison")
plt.xticks(rotation=35, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "model_comparison_metrics.png", dpi=300, bbox_inches="tight")
plt.show()